# Test Pipeline Transferts Libres — PaddleOCR-VL 0.9B

Architecture 2 étapes :
1. **PaddleOCR-VL-1.5** (0.9B) → OCR du texte brut par page  
2. **Regex / règles** → extraction des champs métier + classification du type

Avantages : tourne sur CPU Mac, zéro hallucination, 0.9B = ~2 Go RAM.

**Exécuter les cellules dans l'ordre.**

## 1. Installation des dépendances

In [ ]:
%pip install -q "transformers>=5.0.0" accelerate pillow pymupdf openpyxl
%pip install -q "numpy<2"  # fix numpy 2.x conflict avec torch 2.2
print('✅ Installation OK')

## 2. Imports & environnement

In [ ]:
import sys, os, re, json, time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import torch
import fitz
import numpy as np
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForImageTextToText

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Python     :', sys.version.split()[0])
print('PyTorch    :', torch.__version__)
print('Transformers:', __import__('transformers').__version__)
print('Device     :', DEVICE)

## 3. Configuration

In [ ]:
# ── Modèle ────────────────────────────────────────────────────────────────────
MODEL_ID   = 'PaddlePaddle/PaddleOCR-VL-1.5'   # 0.9B – meilleure qualité OCR

# ── Chemins ───────────────────────────────────────────────────────────────────
INPUT_DIR  = Path('/Users/macbookpro/Desktop/tosyali/projet_tl_one/transfert_in')
OUTPUT_DIR = Path('/Users/macbookpro/Desktop/tosyali/projet_tl_one/paddle_out')
JSON_DIR   = OUTPUT_DIR / 'json'
LOG_PATH   = OUTPUT_DIR / 'pipeline.log'
EXCEL_PATH = OUTPUT_DIR / f'audit_paddle_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

# ── Paramètres PDF ────────────────────────────────────────────────────────────
PDF_ZOOM        = 3.0    # résolution scan
IMAGE_MAX_SIZE  = 2048
BLANK_THRESHOLD = 0.95

# ── Test rapide : limiter le nombre de PDFs ───────────────────────────────────
MAX_PDFS = 5   # mettre None pour tout traiter

pdfs = sorted(INPUT_DIR.glob('*.PDF')) + sorted(INPUT_DIR.glob('*.pdf'))
if MAX_PDFS:
    pdfs = pdfs[:MAX_PDFS]

print(f'Dossiers à traiter : {len(pdfs)}')
for p in pdfs:
    print(f'  {p.name}')

## 4. Chargement du modèle PaddleOCR-VL-1.5

In [ ]:
t0 = time.time()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float32,  # CPU : float32 obligatoire
    low_cpu_mem_usage=True,
)
model.eval()
if DEVICE != 'cpu':
    model = model.to(DEVICE)

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')
print(f'   Type  : {type(model).__name__}')
print(f'   Device: {DEVICE}')

## 5. Utilitaires PDF + OCR

In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    r = max_side / max(w, h)
    return img.resize((int(w * r), int(h * r)), Image.LANCZOS)


def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold


def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc    = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = resize(Image.frombytes('RGB', [pix.width, pix.height], pix.samples))
        pages.append({'index': i, 'image': img})
    doc.close()
    return pages


def ocr_page(image: Image.Image, task: str = 'ocr') -> str:
    """
    Appelle PaddleOCR-VL sur une image.
    task : 'ocr' | 'table' | 'spotting'
    Retourne le texte brut extrait.
    """
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text':  task},
        ]
    }]
    text_in = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_in],
        images=[image],
        return_tensors='pt',
    )
    if DEVICE != 'cpu':
        inputs = {k: v.to(DEVICE) if hasattr(v, 'to') else v for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    generated = output[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


print('✅ Utilitaires OK')

## 6. Normalisation (identique pipeline V12)

In [ ]:
def norm_str(v):
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a', '') else None

def norm_upper(v):
    s = norm_str(v)
    return s.upper() if s else None

def norm_compte(v):
    s = norm_str(v)
    if not s: return None
    return re.sub(r'[^A-Za-z0-9]', '', s).upper()

def norm_montant(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = re.sub(r'[^\d.,]', '', str(v).strip())
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif '.' in s and ',' in s: s = s.replace(',', '')
    elif s.count(',') > 1: s = s.replace(',', '')
    try: return float(s)
    except: return None

def norm_date(v):
    if not v: return None
    s = norm_str(v)
    if not s: return None
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    if m: return f'{m.group(3)}/{m.group(2)}/{m.group(1)}'
    return s

def norm_periode(v):
    if not v: return None
    s = str(v).upper().strip()
    s = re.sub(r'[._\-]', ' ', s)
    s = re.sub(r'PART\s*(\d)', r'PART \1', s)
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Normalisation OK')

## 7. Extracteurs par type de document

PaddleOCR-VL retourne du texte brut → on cherche les champs par regex.

In [ ]:
# ── Détection du type de document ─────────────────────────────────────────────
def detect_type(text: str) -> str:
    t = text.upper()
    if 'ORDRE DE VIREMENT' in t:
        return 'OV'
    if 'ANNEXE II' in t or 'FICHE DE PAIE SPECIALE' in t or 'FICHE DE PAIE SPÉCIALE' in t:
        return 'ANNEXE_II'
    if 'BULLETIN DE PAIE' in t or 'BULLETIN DE SALAIRE' in t:
        return 'BULLETIN'
    if ('ANNEXE I' in t or 'JE SOUSSIGNÉ' in t or 'JE SOUSSIGNE' in t) and 'ANNEXE II' not in t:
        return 'ANNEXE_I'
    return 'AUTRE'


# ── Helpers regex ─────────────────────────────────────────────────────────────
def find_after(text: str, *patterns, nb_words=8):
    """Trouve le texte après le premier pattern correspondant."""
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            rest = text[m.end():].strip()
            words = rest.split()
            return ' '.join(words[:nb_words]) if words else None
    return None


def find_number(text: str, *patterns):
    for pat in patterns:
        m = re.search(pat + r'[\s:]*([\d\s.,]+)', text, re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None


def find_account(text: str):
    """Cherche un numéro de compte bancaire (≥15 chiffres consécutifs ou IBAN)."""
    # IBAN format
    m = re.search(r'[A-Z]{2}\d{2}[A-Z0-9]{10,30}', text)
    if m: return m.group()
    # 20 digits Algerian account
    m = re.search(r'\d{20}', re.sub(r'\s', '', text))
    if m: return m.group()
    # 15+ digits
    m = re.search(r'\d{15,}', re.sub(r'\s', '', text))
    if m: return m.group()
    return None


def find_swift(text: str):
    m = re.search(r'\b([A-Z]{4}[A-Z]{2}[A-Z0-9]{2}(?:[A-Z0-9]{3})?)\b', text)
    return m.group(1) if m else None


def find_date(text: str):
    m = re.search(r'(\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4})', text)
    if m:
        return norm_date(m.group(1).replace('.', '/').replace('-', '/'))
    return None


# ── Extracteur OV ─────────────────────────────────────────────────────────────
def extract_ov(text: str) -> dict:
    d = {'type': 'OV'}

    # Montant chiffres (zone 32)
    m = re.search(r'(?:montant|amount)[\s:]*([\d\s.,]+(?:USD|EUR|DZD)?)', text, re.IGNORECASE)
    if not m:
        m = re.search(r'\b(\d{1,3}(?:[\s.,]\d{3})+(?:[.,]\d{2})?)\b', text)
    d['montant_chiffres'] = norm_montant(m.group(1)) if m else None

    # Monnaie
    m = re.search(r'\b(USD|EUR|GBP|DZD|MAD|TND|CHF)\b', text, re.IGNORECASE)
    d['monnaie'] = m.group(1).upper() if m else None

    # Montant lettres
    m = re.search(r'(?:en\s+toutes\s+lettres|montant\s+en\s+lettres)[\s:]*(.{10,80}?)(?:\n|$)', text, re.IGNORECASE)
    d['montant_lettres'] = norm_str(m.group(1)) if m else None

    # Période (zone 70)
    MOIS = r'(?:JANVIER|FÉVRIER|FEVRIER|MARS|AVRIL|MAI|JUIN|JUILLET|AOÛT|AOUT|SEPTEMBRE|OCTOBRE|NOVEMBRE|DÉCEMBRE|DECEMBRE|JAN|FEV|MAR|AVR|MAI|JUI|JUL|AOU|SEP|OCT|NOV|DEC)'
    m = re.search(MOIS + r'[\s.]*\d{4}', text, re.IGNORECASE)
    d['periode'] = norm_periode(m.group()) if m else None
    m2 = re.search(MOIS, text, re.IGNORECASE)
    d['mois'] = norm_upper(m2.group()) if m2 else None
    m3 = re.search(r'\b(20\d{2})\b', text)
    d['annee'] = m3.group(1) if m3 else None

    # Tranche / complément
    m = re.search(r'\b(P(?:ART(?:IE)?)?\s*[12])\b', text, re.IGNORECASE)
    d['tranche'] = norm_str(m.group(1)) if m else None
    m = re.search(r'\b(COM(?:PL(?:ÉMENT|EMENT)?)?)\b', text, re.IGNORECASE)
    d['complement_ov'] = norm_str(m.group(1)) if m else None

    # Date demande (zone 50)
    d['date_demande'] = find_date(text)

    # Compte donneur d'ordre (zone 50) — N°DOM, Siège Racine Ordinal clé
    m = re.search(r'(?:N°DOM|SIEGE|RACINE|ORDINAL|COMPTE\s*D[EO]NNEUR)[\s:]*([\d\s]{8,25})', text, re.IGNORECASE)
    d['compte_donneur_ordre'] = norm_compte(m.group(1)) if m else None
    if not d['compte_donneur_ordre']:
        d['compte_donneur_ordre'] = find_account(text)

    # Bénéficiaire (zone 59)
    m = re.search(r'(?:BÉNÉFICIAIRE|BENEFICIAIRE|59)[\s:]*(.{5,60}?)(?:\n)', text, re.IGNORECASE)
    d['beneficiaire_nom'] = norm_upper(m.group(1)) if m else None
    m = re.search(r'(?:IBAN|COMPTE\s*BÉNÉFICIAIRE|COMPTE\s*BENEFICIAIRE)[\s:]*([A-Z0-9\s]{15,35})', text, re.IGNORECASE)
    d['beneficiaire_compte'] = norm_compte(m.group(1)) if m else None
    m = re.search(r'(?:ADRESSE|ADDRESS)[\s:]*(.{5,80}?)(?:\n|$)', text, re.IGNORECASE)
    d['beneficiaire_adresse'] = norm_str(m.group(1)) if m else None

    # SWIFT + banque bénéficiaire (zone 57)
    d['code_swift_banque_beneficiaire'] = find_swift(text)
    m = re.search(r'(?:BANQUE|BANK)[\s:]*(.{5,60}?)(?:\n|$)', text, re.IGNORECASE)
    d['nom_banque_beneficiaire'] = norm_upper(m.group(1)) if m else None

    # Nature paiement
    m = re.search(r'\(([^)]{5,60})\)', text)
    d['nature_paiement_autre_libelle'] = norm_str(m.group(1)) if m else None

    return d


# ── Extracteur ANNEXE_II ──────────────────────────────────────────────────────
def extract_annexe2(text: str) -> dict:
    d = {'type': 'ANNEXE_II'}

    MOIS = r'(?:JANVIER|FÉVRIER|FEVRIER|MARS|AVRIL|MAI|JUIN|JUILLET|AOÛT|AOUT|SEPTEMBRE|OCTOBRE|NOVEMBRE|DÉCEMBRE|DECEMBRE|JAN|FEV|MAR|AVR|JUI|JUL|AOU|SEP|OCT|NOV|DEC)'
    m = re.search(r'(?:mois\s+de|mois\s*:)[\s]*(\S+(?:\s+\d{4})?)', text, re.IGNORECASE)
    d['mois_transfert'] = norm_periode(m.group(1)) if m else None

    # Nom
    m = re.search(r'(?:nom\s+et\s+prénom|nom\s+prenom|travailleur)[\s:]*(.{3,60}?)(?:\n|$)', text, re.IGNORECASE)
    d['nom_prenom_travailleur'] = norm_upper(m.group(1)) if m else None

    # Compte local (20 chiffres)
    m = re.search(r'(?:compte\s+bancaire|compte\s+local)[\s:]*([\d\s]{20,25})', text, re.IGNORECASE)
    raw = norm_compte(m.group(1)) if m else None
    if not raw:
        c = find_account(text)
        raw = norm_compte(c) if c else None
    d['compte_bancaire_local'] = raw

    # Salaire
    m = re.search(r'(?:salaire\s+mensuel|salaire\s+brut|traitement)[\s:]*([\d\s.,]+)', text, re.IGNORECASE)
    d['salaire_mensuel'] = norm_montant(m.group(1)) if m else None

    # Jours
    m = re.search(r'nombre\s+de\s+jour(?:s)?(?:\s+travaillés?)?[\s:]*(\d+)', text, re.IGNORECASE)
    d['nombre_jours'] = norm_str(m.group(1)) if m else None
    m = re.search(r"nombre\s+de\s+jour(?:s)?\s+d'absence[\s:]*(\d+)", text, re.IGNORECASE)
    d['nombre_jours_absence'] = norm_str(m.group(1)) if m else None

    # Part transférable
    m = re.search(r'(?:part\s+transf[eé]rable|montant\s+transf[eé]r)[\s:]*([\d\s.,]+)', text, re.IGNORECASE)
    d['part_transferable'] = norm_montant(m.group(1)) if m else None

    # Pays destination
    m = re.search(r'(?:pays\s+de\s+destination|pays\s+destination)[\s:]*(.{3,40}?)(?:\n|$)', text, re.IGNORECASE)
    d['pays_destination'] = norm_upper(m.group(1)) if m else None

    # Compte devise étranger — IBAN
    m = re.search(r'(?:compte\s+devise|iban)[\s:]*([A-Z]{2}\d{2}[A-Z0-9\s]{10,30})', text, re.IGNORECASE)
    raw_iban = norm_compte(m.group(1)) if m else None
    d['numero_compte_devise_etranger'] = raw_iban

    # Banque étrangère (avant l'IBAN)
    m = re.search(r'compte\s+devise\s*:[\s]*(.{3,60}?)\s+(?:[A-Z]{2}\d{2}|\d{10})', text, re.IGNORECASE)
    d['nom_banque_etranger'] = norm_str(m.group(1)) if m else None

    # Domiciliation
    m = re.search(r'(?:DOMICILIATION|N°DOM)[\s:]*([\d|A-Za-z.\-\s]{5,60}?)(?:\n|$)', text, re.IGNORECASE)
    d['numero_domiciliation'] = norm_str(m.group(1)) if m else None

    return d


# ── Extracteur ANNEXE_I ───────────────────────────────────────────────────────
def extract_annexe1(text: str) -> dict:
    d = {'type': 'ANNEXE_I'}

    m = re.search(r'(?:je\s+soussigné[e]?)[\s,]*(.{3,80}?)(?:\n|,)', text, re.IGNORECASE)
    d['nom_prenom_employe'] = norm_upper(m.group(1)) if m else None

    m = re.search(r'(?:né[e]?\s+le|date\s+de\s+naissance)[\s:]*([\d/\-.]{8,10})', text, re.IGNORECASE)
    d['date_naissance'] = norm_date(m.group(1).replace('-', '/')) if m else None

    m = re.search(r'(?:résidence|domicile|adresse)[\s:]*(.{5,80}?)(?:\n|$)', text, re.IGNORECASE)
    d['résidence'] = norm_str(m.group(1)) if m else None

    c = find_account(text)
    d['compte_bancaire_local'] = norm_compte(c) if c else None

    m = re.search(r'(?:responsable|signataire|directeur)[\s:]*(.{3,80}?)(?:\n|$)', text, re.IGNORECASE)
    d['nom_prenom_signataire'] = norm_upper(m.group(1)) if m else None

    m = re.search(r'(?:bethioua\s+le|fait\s+le|le\s+:?)[\s]*([\d/\-.]{8,10})', text, re.IGNORECASE)
    d['date_signature'] = norm_date(m.group(1).replace('-', '/')) if m else None

    return d


# ── Extracteur BULLETIN ───────────────────────────────────────────────────────
def extract_bulletin(text: str) -> dict:
    d = {'type': 'BULLETIN'}

    m = re.search(r'(?:nom\s+et\s+prénom|salarié|employé)[\s:]*(.{3,60}?)(?:\n|$)', text, re.IGNORECASE)
    d['nom_prenom_salarie'] = norm_upper(m.group(1)) if m else None

    m = re.search(r'(?:matricule|mat\.?)[\s:]*([A-Z0-9\-]{4,15})', text, re.IGNORECASE)
    d['matricule'] = norm_str(m.group(1)) if m else None

    MOIS = r'(?:JANVIER|FÉVRIER|FEVRIER|MARS|AVRIL|MAI|JUIN|JUILLET|AOÛT|AOUT|SEPTEMBRE|OCTOBRE|NOVEMBRE|DÉCEMBRE|DECEMBRE)'
    m = re.search(MOIS + r'[\s.]*\d{4}', text, re.IGNORECASE)
    d['mois_bulletin'] = norm_periode(m.group()) if m else None

    def montant_apres(label):
        m = re.search(label + r'[\s:]*([\d\s.,]+)', text, re.IGNORECASE)
        return norm_montant(m.group(1)) if m else None

    d['salaire_base']      = montant_apres(r'salaire\s+de\s+base')
    d['salaire_brut']      = montant_apres(r'salaire\s+brut')
    d['retenue_ss']        = montant_apres(r'(?:sécurité\s+sociale|ss|cotisation\s+ss)')
    d['retenue_irg']       = montant_apres(r'(?:irg|impôt\s+sur\s+le\s+revenu)')
    d['retenue_mutuelle']  = montant_apres(r'mutuelle')
    d['net_a_payer']       = montant_apres(r'net\s+[àa]\s+payer')

    return d


# ── Dispatcher ────────────────────────────────────────────────────────────────
def extract_fields(doc_type: str, text: str) -> dict:
    if doc_type == 'OV':       return extract_ov(text)
    if doc_type == 'ANNEXE_II': return extract_annexe2(text)
    if doc_type == 'ANNEXE_I': return extract_annexe1(text)
    if doc_type == 'BULLETIN': return extract_bulletin(text)
    return {'type': doc_type}


print('✅ Extracteurs OK')

## 8. Test unitaire sur 1 page (debug)

In [ ]:
# Prend la première page du premier PDF pour vérifier le texte brut OCR
first_pdf = pdfs[0]
print(f'Test sur : {first_pdf.name}')

pages = pdf_to_pages(first_pdf)
non_blanks = [p for p in pages if not is_blank(p['image'])]
print(f'Pages non vides : {len(non_blanks)}')

# OCR de la première page non vide
t0 = time.time()
ocr_text = ocr_page(non_blanks[0]['image'], task='ocr')
print(f'OCR en {time.time()-t0:.1f}s')

print('\n' + '='*60)
print('TEXTE OCR BRUT')
print('='*60)
print(ocr_text)
print('='*60)

# Détection + extraction
doc_type = detect_type(ocr_text)
print(f'\nType détecté : {doc_type}')
fields = extract_fields(doc_type, ocr_text)
print(json.dumps(fields, ensure_ascii=False, indent=2, default=str))

## 8b. Test OCR table (si ANNEXE_II)
Si la page contient un tableau, lancer aussi la tâche `table` pour capturer le tableau de domiciliation.

In [ ]:
# Ne lancer que si le document est ANNEXE_II pour tester la tâche 'table'
# Changer page_idx pour tester une page différente
page_idx = 0

t0 = time.time()
table_text = ocr_page(non_blanks[page_idx]['image'], task='table')
print(f'Table OCR en {time.time()-t0:.1f}s')
print('='*60)
print(table_text)
print('='*60)

## 9. Pipeline complet — batch de PDFs

In [ ]:
import gc

TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

def log(msg: str):
    ligne = f"{datetime.now().strftime('%H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')


def process_pdf(pdf_path: Path, verbose=True) -> dict:
    pages       = pdf_to_pages(pdf_path)
    results     = {}
    doublons    = []
    total_pages = 0
    t_dossier   = time.time()

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    for page in pages:
        if is_blank(page['image']):
            continue

        t_page = time.time()
        ocr_text = ocr_page(page['image'], task='ocr')
        total_pages += 1

        doc_type = detect_type(ocr_text)

        if doc_type == 'AUTRE':
            if verbose:
                print(f'  Page {page["index"]+1} → AUTRE ({time.time()-t_page:.1f}s) — ignorée')
            continue

        # Pour ANNEXE_II : ajouter une passe 'table' pour domiciliation
        extra_text = ''
        if doc_type == 'ANNEXE_II':
            table_text = ocr_page(page['image'], task='table')
            extra_text = '\n' + table_text

        data = extract_fields(doc_type, ocr_text + extra_text)

        if verbose:
            print(f'  Page {page["index"]+1} → {doc_type:10s} ({time.time()-t_page:.1f}s)')
            for k, v in data.items():
                if k != 'type' and v is not None:
                    print(f'    {k:30s}: {v}')

        if doc_type in results:
            doublons.append(doc_type)
            if verbose:
                print(f'    ⚠️  DOUBLON {doc_type}')
            continue

        results[doc_type] = data

    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'fichier':          pdf_path.name,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'temps_total_s':    round(time.time() - t_dossier, 2),
        'pages_actives':    total_pages,
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }


# ── Exécution ─────────────────────────────────────────────────────────────────
deja_traites = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter    = [p for p in pdfs if p.stem not in deja_traites]

log(f'PDFs total   : {len(pdfs)}')
log(f'Déjà traités : {len(deja_traites)}')
log(f'À traiter    : {len(a_traiter)}')

rows   = []
n_ok   = 0
n_err  = 0
t_total = time.time()

for num, pdf_path in enumerate(a_traiter, start=1):
    try:
        result = process_pdf(pdf_path, verbose=True)

        json_file = JSON_DIR / f'{pdf_path.stem}.json'
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)

        rows.append(result)
        n_ok += 1
        msg = (f'[{num:>3}/{len(a_traiter)}] ✅ {pdf_path.name}'
               f' | {result["temps_total_s"]}s'
               f' | {result["pages_trouvees"]}')
        if result.get('pages_manquantes'):
            msg += f' | ⚠️ manque: {result["pages_manquantes"]}'
        log(msg)

    except Exception as e:
        n_err += 1
        log(f'[{num:>3}/{len(a_traiter)}] ❌ {pdf_path.name} — {e}')
        import traceback; traceback.print_exc()

    gc.collect()

elapsed = time.time() - t_total
log(f'✅ Terminé en {elapsed:.1f}s | OK: {n_ok} | Erreurs: {n_err}')

## 10. Export Excel

In [ ]:
SCHEMA = {
    'META': [
        ('fichier',           'Fichier'),
        ('date_traitement',   'Date traitement'),
        ('temps_total_s',     'Temps (s)'),
        ('pages_actives',     'Pages actives'),
        ('pages_trouvees',    'Pages trouvées'),
        ('pages_manquantes',  'Pages manquantes'),
        ('anomalies',         'Anomalies'),
    ],
    'OV': [
        ('monnaie',                       'Monnaie'),
        ('montant_chiffres',              'Montant (chiffres)'),
        ('montant_lettres',               'Montant (lettres)'),
        ('periode',                       'Période'),
        ('mois',                          'Mois'),
        ('annee',                         'Année'),
        ('tranche',                       'Tranche'),
        ('complement_ov',                 'Complément'),
        ('date_demande',                  'Date demande'),
        ('compte_donneur_ordre',          'Compte donneur ordre'),
        ('nature_paiement_autre_libelle', 'Devise transfert'),
        ('beneficiaire_nom',              'Bénéficiaire nom'),
        ('beneficiaire_compte',           'Bénéficiaire compte'),
        ('beneficiaire_adresse',          'Bénéficiaire adresse'),
        ('code_swift_banque_beneficiaire','SWIFT'),
        ('nom_banque_beneficiaire',       'Banque bénéficiaire'),
    ],
    'ANNEXE_I': [
        ('nom_prenom_employe',    'Nom Prénom employé'),
        ('date_naissance',        'Date naissance'),
        ('résidence',             'Résidence'),
        ('compte_bancaire_local', 'Compte bancaire'),
        ('nom_prenom_signataire', 'Nom Prénom signataire'),
        ('date_signature',        'Date signature'),
    ],
    'ANNEXE_II': [
        ('mois_transfert',                 'Mois transfert'),
        ('nom_prenom_travailleur',          'Nom Prénom'),
        ('compte_bancaire_local',           'Compte bancaire'),
        ('salaire_mensuel',                 'Salaire mensuel'),
        ('nombre_jours',                    'Nombre jours'),
        ('nombre_jours_absence',            'Jours absence'),
        ('part_transferable',               'Part transférable'),
        ('pays_destination',                'Pays destination'),
        ('nom_banque_etranger',             'Banque étrangère'),
        ('numero_compte_devise_etranger',   'Compte étranger'),
        ('numero_domiciliation',            'N° Domiciliation'),
    ],
    'BULLETIN': [
        ('nom_prenom_salarie',  'Nom Prénom'),
        ('matricule',           'Matricule'),
        ('mois_bulletin',       'Mois bulletin'),
        ('salaire_base',        'Salaire base'),
        ('salaire_brut',        'Salaire brut'),
        ('retenue_ss',          'Retenue SS'),
        ('retenue_irg',         'Retenue IRG'),
        ('retenue_mutuelle',    'Retenue mutuelle'),
        ('net_a_payer',         'Net à payer'),
    ],
}

COLORS = {
    'META':      {'header': 'FF1F4E79', 'col': 'FFD6E4F0'},
    'OV':        {'header': 'FF833C00', 'col': 'FFFCE4D6'},
    'ANNEXE_I':  {'header': 'FF375623', 'col': 'FFE2EFDA'},
    'ANNEXE_II': {'header': 'FF203864', 'col': 'FFDAE3F3'},
    'BULLETIN':  {'header': 'FF3F3151', 'col': 'FFEDE7F6'},
}


def create_excel(path: Path, rows: list):
    wb = Workbook()
    ws = wb.active
    ws.title = 'Dossiers'

    all_cols = []
    for groupe, cols in SCHEMA.items():
        for field_key, label in cols:
            all_cols.append((groupe, field_key, label))

    col_idx  = 1
    group_map = defaultdict(list)
    for groupe, _, _ in all_cols:
        group_map[groupe].append(col_idx)
        col_idx += 1

    for groupe, cols in group_map.items():
        s, e = cols[0], cols[-1]
        if s < e:
            ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s)
        c.value = groupe
        c.font  = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['header'])
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22

    for i, (groupe, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i)
        c.value = label
        c.font  = Font(bold=True, name='Arial', size=9)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        ws.column_dimensions[get_column_letter(i)].width = 20
    ws.row_dimensions[2].height = 35
    ws.freeze_panes = ws.cell(row=3, column=len(SCHEMA['META']) + 1)

    for row_num, dossier in enumerate(rows, start=3):
        for col_idx, (groupe, field_key, _) in enumerate(all_cols, start=1):
            val = (dossier.get(field_key) if groupe == 'META'
                   else (dossier.get(groupe) or {}).get(field_key))
            c = ws.cell(row=row_num, column=col_idx)
            c.value = val
            c.font  = Font(name='Arial', size=9)
            c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
            if isinstance(val, float):
                c.number_format = '0.00'
            if groupe == 'META' and field_key in ('anomalies', 'pages_manquantes') and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')

    wb.save(path)
    print(f'✅ Excel : {path}')
    print(f'   {len(rows)} dossiers | {len(all_cols)} colonnes')


# Charger tous les JSON (y compris anciens)
all_rows = []
for json_file in sorted(JSON_DIR.glob('*.json')):
    with open(json_file, encoding='utf-8') as f:
        all_rows.append(json.load(f))

create_excel(EXCEL_PATH, all_rows)

## 11. Analyse rapide des résultats

In [ ]:
print(f'\n📊 {len(all_rows)} dossiers traités')
print()

manquants  = [r for r in all_rows if r.get('pages_manquantes')]
anomalies  = [r for r in all_rows if r.get('anomalies')]
complets   = [r for r in all_rows if not r.get('pages_manquantes')]
temps_moy  = sum(r['temps_total_s'] for r in all_rows) / max(1, len(all_rows))

print(f'✅ Complets       : {len(complets)}/{len(all_rows)}')
print(f'⚠️  Incomplets    : {len(manquants)}')
print(f'🔴 Anomalies      : {len(anomalies)}')
print(f'⏱  Temps moyen   : {temps_moy:.1f}s/dossier')
print()

if manquants:
    print('Dossiers incomplets :')
    for r in manquants:
        print(f'  {r["fichier"]:50s} manque: {r["pages_manquantes"]}')

print(f'\nExcel : {EXCEL_PATH}')